# FastAPI Moderatiesysteem – Backend API

## Doel van de Applicatie

Deze applicatie vormt de backend van het AI-moderatiesysteem. Het systeem stelt gebruikers in staat om afbeeldingen te uploaden, waarna een getraind ResNet18-model bepaalt of de afbeelding als **safe** of **unsafe** wordt geclassificeerd.

Naast automatische classificatie bevat het systeem ook:

* Uploadfunctionaliteit
* Logging van voorspellingen
* Feedbackmechanisme voor gebruikers
* Geheugen voor veilige en onveilige afbeeldingen
* Hash-gebaseerde herkenning van eerder gedetecteerde unsafe afbeeldingen
* Gezondheidscontrole van de applicatie

Hierdoor ontstaat een moderatiesysteem dat niet alleen voorspellingen uitvoert, maar ook kan leren van menselijke feedback.

---

# Importeren van Bibliotheken

```python
from fastapi import FastAPI, UploadFile, File, Request
from fastapi.responses import HTMLResponse, JSONResponse
from fastapi.templating import Jinja2Templates
from fastapi.staticfiles import StaticFiles
```

Deze modules vormen de basis van de FastAPI-webapplicatie.

### Functionaliteit

| Module          | Doel                             |
| --------------- | -------------------------------- |
| FastAPI         | Webframework                     |
| UploadFile      | Uploaden van bestanden           |
| JSONResponse    | JSON-antwoorden versturen        |
| HTMLResponse    | HTML-pagina's tonen              |
| Jinja2Templates | Dynamische HTML-rendering        |
| StaticFiles     | Serveren van statische bestanden |

---

# Initialiseren van de Applicatie

```python
app = FastAPI()
```

## Wat gebeurt hier?

Een nieuwe FastAPI-applicatie wordt aangemaakt.

Deze applicatie beheert alle API-routes en verzoeken.

---

# Configuratie van Mappen en Bestanden

```python
UPLOAD_FOLDER = "uploads"
STATIC_FOLDER = "static"

SAFE_MEMORY = "memory/safe"
UNSAFE_MEMORY = "memory/unsafe"

MODEL_PATH = "model/resnet18_sketch_model_best.pth"

LOG_FILE = "moderation_log.json"
```

## Doel van de Mappen

### uploads/

Hier worden geüploade afbeeldingen opgeslagen.

---

### memory/safe/

Bevat afbeeldingen die door gebruikers als veilig zijn bevestigd.

---

### memory/unsafe/

Bevat afbeeldingen die door gebruikers als onveilig zijn bevestigd.

---

### model/

Bevat het getrainde ResNet18-model.

---

### moderation_log.json

Slaat alle moderatieacties op.

---

# Automatisch Aanmaken van Mappen

```python
os.makedirs(
    UPLOAD_FOLDER,
    exist_ok=True
)
```

## Waarom?

Wanneer een map nog niet bestaat wordt deze automatisch aangemaakt.

Dit voorkomt fouten tijdens het uitvoeren van de applicatie.

---

# Static Files Configureren

```python
app.mount(
    "/uploads",
    StaticFiles(directory=UPLOAD_FOLDER),
    name="uploads"
)
```

## Wat gebeurt hier?

Geüploade afbeeldingen worden toegankelijk via een URL.

Voorbeeld:

```text
/uploads/voorbeeld.jpg
```

Hierdoor kunnen afbeeldingen worden weergegeven in de webinterface.

---

# Jinja2 Templates

```python
templates = Jinja2Templates(
    directory="templates"
)
```

## Functie

De HTML-pagina's worden dynamisch gegenereerd met behulp van Jinja2.

Dit maakt het mogelijk om bijvoorbeeld moderatielogs automatisch op de website te tonen.

---

# Logbestand Initialiseren

```python
if not os.path.exists(LOG_FILE):

    with open(LOG_FILE, "w") as f:

        json.dump([], f)
```

## Wat gebeurt hier?

Wanneer het logbestand nog niet bestaat wordt een lege JSON-lijst aangemaakt.

Voorbeeld:

```json
[]
```

---

# Geblokkeerde Woorden

```python
BLOCKED_WORDS = [
    "sex",
    "nazi",
    "rape",
    "terrorist",
    "kill",
    "suicide",
    "slur",
    "nsfw"
]
```

## Waarom?

Voordat een afbeelding door het AI-model wordt geanalyseerd wordt eerst gekeken naar de bestandsnaam.

Wanneer een verboden woord voorkomt in de bestandsnaam kan de afbeelding direct als unsafe worden gemarkeerd.

Voorbeeld:

```text
nazi_picture.jpg
```

Resultaat:

```text
unsafe
```

---

# Herkennen van Bekende Unsafe Afbeeldingen

```python
KNOWN_UNSAFE_HASHES = set()
```

## Wat is een Hash?

Een hash is een unieke digitale vingerafdruk van een bestand.

Voorbeeld:

```python
def get_image_hash(image_bytes):

    return hashlib.md5(
        image_bytes
    ).hexdigest()
```

Elke afbeelding krijgt een unieke MD5-hash.

---

## Waarom nuttig?

Wanneer een eerder gedetecteerde unsafe afbeelding opnieuw wordt geüpload:

* hoeft het model niet opnieuw te classificeren
* wordt de afbeelding direct geblokkeerd

Dit verhoogt de efficiëntie van het systeem.

---

# Laden van het AI-Model

```python
model = models.resnet18(
    weights=None
)
```

Hier wordt een ResNet18-model aangemaakt.

---

## Aanpassen van de Uitvoerlaag

```python
model.fc = nn.Linear(
    model.fc.in_features,
    2
)
```

Het model wordt aangepast voor twee klassen:

| Klasse |
| ------ |
| safe   |
| unsafe |

---

## Laden van de Gewichten

```python
model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=torch.device("cpu")
    )
)
```

Hier worden de eerder getrainde modelgewichten geladen.

Hierdoor hoeft het model niet opnieuw getraind te worden bij iedere start van de applicatie.

---

# Afbeelding Preprocessing

```python
transform = transforms.Compose([
```

Voordat een afbeelding naar het model gaat wordt deze eerst verwerkt.

---

## Grayscale naar RGB

```python
transforms.Grayscale(
    num_output_channels=3
)
```

ResNet18 verwacht drie kleurkanalen.

De sketch-afbeeldingen worden daarom omgezet naar drie identieke kanalen.

---

## Resize

```python
transforms.Resize(
    (224,224)
)
```

Alle afbeeldingen worden geschaald naar het formaat dat ResNet18 verwacht.

---

## Tensor Conversie

```python
transforms.ToTensor()
```

Zet afbeeldingen om naar PyTorch tensors.

---

## Normalisatie

```python
Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)
```

Gebruikt dezelfde normalisatie als tijdens de ImageNet-training.

---

# Labels

```python
labels = [
    "safe",
    "unsafe"
]
```

De modeluitvoer wordt gekoppeld aan leesbare labels.

---

# Homepage Route

```python
@app.get("/")
```

## Functie

Toont de webinterface.

Hier worden:

* uploads
* classificaties
* moderatielogs

zichtbaar gemaakt.

---

# Afbeelding Classificeren

```python
@app.post("/predict")
```

Dit is de belangrijkste endpoint van het systeem.

---

## Stap 1: Upload Ontvangen

```python
image_bytes = await file.read()
```

De afbeelding wordt ingelezen.

---

## Stap 2: Hash Genereren

```python
image_hash = get_image_hash(
    image_bytes
)
```

Een unieke identificatie van de afbeelding wordt gemaakt.

---

## Stap 3: Controle op Bekende Unsafe Afbeeldingen

```python
if image_hash in KNOWN_UNSAFE_HASHES:
```

Wanneer de afbeelding eerder als unsafe werd gemarkeerd:

```json
{
  "prediction": "unsafe"
}
```

wordt direct teruggegeven.

---

## Stap 4: Bestandsnaamfilter

```python
for word in BLOCKED_WORDS:
```

De bestandsnaam wordt gecontroleerd op verboden woorden.

Wanneer een match wordt gevonden:

* wordt de afbeelding opgeslagen in `memory/unsafe`
* wordt de hash onthouden
* wordt direct een unsafe-classificatie teruggegeven

---

## Stap 5: Upload Opslaan

```python
with open(filepath, "wb") as f:
```

De afbeelding wordt opgeslagen in de uploadmap.

---

## Stap 6: AI-Classificatie

```python
outputs = model(image_tensor)
```

De afbeelding wordt door het ResNet18-model verwerkt.

---

## Stap 7: Kansberekening

```python
probabilities = torch.softmax(
    outputs,
    dim=1
)
```

De modeluitvoer wordt omgezet naar waarschijnlijkheden.

---

## Stap 8: Hoogste Score Kiezen

```python
confidence, predicted = torch.max(
    probabilities,
    1
)
```

Het label met de hoogste waarschijnlijkheid wordt geselecteerd.

---

# Confidence Threshold

```python
if confidence_score < 0.75:
    prediction = "review_required"
```

## Waarom?

Wanneer het model onzeker is wordt geen definitieve beslissing genomen.

De afbeelding wordt dan gemarkeerd als:

```text
review_required
```

Dit vermindert foutieve moderatiebeslissingen.

---

# Logging van Resultaten

Voor iedere classificatie wordt een logrecord opgeslagen.

Voorbeeld:

```json
{
    "filename": "...",
    "prediction": "safe",
    "confidence": 0.92
}
```

---

# Feedbackmechanisme

```python
@app.post("/feedback/{filename}/{label}")
```

## Doel

Gebruikers kunnen aangeven of de voorspelling correct was.

---

## Safe Feedback

Afbeelding wordt opgeslagen in:

```text
memory/safe
```

---

## Unsafe Feedback

Afbeelding wordt opgeslagen in:

```text
memory/unsafe
```

---

## Waarom belangrijk?

Deze afbeeldingen kunnen later opnieuw worden gebruikt voor training.

Hierdoor ontstaat een vorm van continue verbetering van het model.

---

# Verwijderen van Logs

```python
@app.delete("/delete-log/{filename}")
```

## Functie

Verwijdert:

* het logrecord
* de geüploade afbeelding

---

# Logs Ophalen

```python
@app.get("/logs")
```

Geeft alle moderatiegeschiedenis terug in JSON-formaat.

Dit kan worden gebruikt door de frontend.

---

# Health Check Endpoint

```python
@app.get("/health")
```

## Functie

Controleert of de applicatie actief is.

Resultaat:

```json
{
    "status": "running"
}
```

Dit endpoint wordt vaak gebruikt voor monitoring en deploymentplatformen.

---

# Conclusie

Deze FastAPI-applicatie vormt de backend van het moderatiesysteem en verbindt het getrainde ResNet18-model met een gebruiksvriendelijke webinterface. Het systeem verwerkt geüploade afbeeldingen, voert automatische classificatie uit en slaat alle resultaten op in een logbestand. Daarnaast bevat het extra beveiligingslagen zoals bestandsnaamfiltering, hash-gebaseerde herkenning van eerder gedetecteerde unsafe content en een confidence-threshold voor onzekere voorspellingen. Door het geïntegreerde feedbackmechanisme kunnen gebruikers classificaties corrigeren, waarna deze voorbeelden worden opgeslagen voor toekomstige modeltraining. Hierdoor ontstaat een zelfverbeterend moderatiesysteem dat zowel automatische detectie als menselijke validatie combineert.
